1. 쓱닷컴 사이트에 들어간다
- 제일 처음에 뭘 해야할지 생각해보고, 생각이 안나면 수업한 내용을 찾아본다
- 그래도 모르겠다면 검색해서 찾아본다
- 그래도 모르겠자면 ai에게 물어보는데 이땐 해달라고 하지말고 공부할 수 있게끔 질문한다
   
2. 신세계몰에 들어간다
3. 쓱-특가에 들어간다
4. 식품에 들어간다
5. 식품 50개의 타이틀, 할인율을 수집한다
- 상품들을 감싸안고 있는 요소를 확인
- 해당 요소 안에 실제 각 상품별 요소를 확인
- 각 상품별 요소 안에 상품명, 할인률, 판매금액, url, 링크 요소를 확인

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import csv

URL = "https://www.ssg.com/"

service = Service(ChromeDriverManager().install())
options = Options()

options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(2)

# 버튼 찾아서 누르는 코드

# shinsegaemall_tab = diver.find_elements(By.CSS_SELECTOR, "a.gnb_mall_link.clickable")[1] # 첫번쨰방법
shinsegaemall_tab = driver.find_element(By.XPATH, "//a[contains(@class, 'gnb_mall_link') and contains(text(), '신세계몰')]") # 두번째방법
shinsegaemall_tab.click()
time.sleep(2)

ssgspecial_tab = driver.find_element(By.XPATH, "//a[contains(@class, 'menu_lnk') and contains(text(), '쓱-특가')]") 
ssgspecial_tab.click()
time.sleep(2)

food_button = driver.find_element(By.XPATH, "//button[@data-index='8']") 
food_button.click()
time.sleep(2)


driver.quit()

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.keys import Keys 
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import csv
import re

URL = "https://www.ssg.com/"

service = Service(ChromeDriverManager().install())
options = Options()

options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("--start-maximized")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36")
options.add_argument("--lang=ko_KR")
options.add_argument("--no-sandbox")

driver = webdriver.Chrome(service=service, options=options)
driver.get(URL)
time.sleep(2)


shinsegaemall_tab = driver.find_element(By.XPATH, "//a[contains(@class, 'gnb_mall_link') and contains(text(), '신세계몰')]")
shinsegaemall_tab.click()
time.sleep(2)

ssgspecial_tab = driver.find_element(By.XPATH, "//a[contains(@class, 'menu_lnk') and contains(text(), '쓱-특가')]") 
ssgspecial_tab.click()
time.sleep(2)

food_button = driver.find_element(By.XPATH, "//button[@data-index='8']") 
food_button.click()
time.sleep(2)

# 개별상품 크롤링 코드
results = list()
seen_urls = set() # 중복값을 허용하지 않기 위해 썼다. 이름, 가격등은 중복이 될 수 있기 때문에 절대 중복 가능성이 없는 url로 선택

def clean_text(s) :
    if not s :
        return "" # 만약 s가 없다면 빈칸 ("")으로 반환해라
    s = re.sub(r"\s+", " ", s) # s에서 1개 이상의 (+) 이스케이프 시쿼스 (\s)가 나온다면 한칸만 띄어서 표시 (" ")해라
    return s.strip() # 반환되는 값을 strip으로 정리해랴

while len(results) < 50 : # 그냥 [:50]을 할수도 있지만 다른 반복문도 써보자
    cards = driver.find_elements(By.CSS_SELECTOR, "div.template-grid-item.css-8kawlz")
    added_this_round = 0
    
    for index, card in enumerate(cards) : 
    
    # for card in cards : # url찾아오기
        try : 
            a = card.find_element(By.CSS_SELECTOR, "a[href]")
            url = a.get_attribute("href")
        except :
            continue
            
        try : # 상품명 찾아오기
            name = card.find_element(By.CSS_SELECTOR, "p.chakra-text.css-19bfb2a").text.strip()
            name = clean_text(name)
        except :
            name = ""

        saleAndprice = card.find_element(By.CSS_SELECTOR, "div.chakra-stack.css-ffjhre")

        try : 
            discount = saleAndprice.find_element(By.CSS_SELECTOR, "em.css-aywnvu").text.strip()
            discount = clean_text(discount).replace("할인율", "").strip()
        except :
            discount = ""
        try : 
            price = saleAndprice.find_element(By.CSS_SELECTOR, "em.css-1oiygnj").text.strip()
            price = clean_text(price).replace("판매가격", "").strip()
        except :
            price = ""

        if url in seen_urls :
            continue

        seen_urls.add(url) # 여기까지만 쓰면 값이 50개도 안되고, 중복값도 허용하지 않아서 무한루프에 빠진다 그래서 아래와 같이 트릭을 사용 (아래 if문)
        
        results.append({"No": index + 1, "상품명": name, "판매금액": price, "할인률": discount, "URL": url})
        added_this_round += 1

    if added_this_round == 0 : # 더 이상 새로운 값이 추가되지 않는다면
        print("새 상품이 더 이상 로드되지 않습니다")
        break
df = pd.DataFrame(results).head(50)
driver.quit()

df

새 상품이 더 이상 로드되지 않습니다


,No,상품명,판매금액,할인률,URL
0,1,오니스트나를위한 아름다움의 시작 10%추가할인+증정헤택,"62,757원외",~10%,https://shinsegaemall.ssg.com/item/dealItemVie...
1,2,농협안심한우농협안심한우/한돈 인기전 ~45% 할인 특가,"8,498원외",~45%,https://shinsegaemall.ssg.com/item/dealItemVie...
2,3,산과들에캐슈넛/하루견과 외 인기 BEST 할인,"16,800원외",~9%,https://shinsegaemall.ssg.com/item/dealItemVie...
3,4,엘빈즈엘빈즈 이유식 30팩 배도라지즙 10입 증정,"61,850원",5%,https://shinsegaemall.ssg.com/item/itemView.ss...
4,5,감동썬키스트 고당도 만다린 쓱 멤버십 특가할인,"15,900원",20%,https://shinsegaemall.ssg.com/item/itemView.ss...
5,6,다농이네다농이네 과일GIFT 프리미엄세트 할인,"39,800원외",,https://shinsegaemall.ssg.com/item/dealItemVie...
6,7,달찐과일달찐과일 선물세트 나주배/사과 등 GIFT할인,"27,900원외",,https://shinsegaemall.ssg.com/item/dealItemVie...
7,8,대천김곱창 캔김/도시락김 외 인기 BEST 특가,"51,000원외",,https://shinsegaemall.ssg.com/item/dealItemVie...
8,9,SSG정담상주감도가 곶감GIFT BEST상품 모음,"35,900원외",~35%,https://shinsegaemall.ssg.com/item/dealItemVie...
9,10,신세계푸드김치/베이커리/국탕류 ~40%할인,"25,900원외",~35%,https://shinsegaemall.ssg.com/item/dealItemVie...
